In [1]:
import json
import random
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

In [2]:
import sys

ROOT = Path('/scratch/network/cr7998/cv_emergence_project')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

FB         = ROOT / 'data' / 'FunnyBirds'
BASE_FEATS = ROOT / 'features' / 'resnet50_funnybirds'
CBM_FEATS  = ROOT / 'features' / 'resnet50_cbm_funnybirds'

assert FB.exists(),           f'Missing FunnyBirds folder: {FB}'
assert (FB / 'dataset_train.json').exists(), f'Missing dataset_train.json'

for name, d in [('baseline', BASE_FEATS), ('cbm', CBM_FEATS)]:
    if not d.exists(): print(f'  [warn] {name} features not found: {d}')
    else:              print(f'  [ok]   {name}: {d}')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device: {device}')

  [ok]   baseline: /scratch/network/cr7998/cv_emergence_project/features/resnet50_funnybirds
  [ok]   cbm: /scratch/network/cr7998/cv_emergence_project/features/resnet50_cbm_funnybirds
device: cpu


In [3]:
def load_species_maps(fb_root: Path):
    """
    Load species ID -> name mappings from metadata/classes.csv.
    Mirrors load_species_maps(cub_root) in recall.ipynb.
    """
    classes_csv = fb_root / 'metadata' / 'classes.csv'
    if not classes_csv.exists():
        raise FileNotFoundError(
            'metadata/classes.csv not found. Run prepare_funnybirds_metadata.py first.')
    df = pd.read_csv(classes_csv)
    id2name = dict(zip(df['class_id'], df['class_name']))
    return id2name


def load_meta(fb_root: Path) -> pd.DataFrame:
    """
    Load image metadata with species information.
    Returns DataFrame: image_id, species_id, species_name, is_train.
    Mirrors load_meta(cub_root) in recall.ipynb.
    """
    images_csv = fb_root / 'metadata' / 'images.csv'
    if not images_csv.exists():
        raise FileNotFoundError(
            'metadata/images.csv not found. Run prepare_funnybirds_metadata.py first.')
    df = pd.read_csv(images_csv)
    id2name = load_species_maps(fb_root)
    df['species_id']   = df['class_id']
    df['species_name'] = df['class_id'].map(id2name)
    return df


def load_image_attr_labels_robust(fb_root: Path) -> pd.DataFrame:
    """
    Load per-image binary concept labels from metadata/image_concepts_binary.csv.
    Returns long-form DataFrame: image_id, attr_id, attr_name, is_present, certainty.
    certainty = 1 always (FunnyBirds labels are ground-truth, no annotation noise).
    Mirrors load_image_attr_labels_robust(cub_root) in recall.ipynb.
    """
    concepts_csv = fb_root / 'metadata' / 'image_concepts_binary.csv'
    if not concepts_csv.exists():
        raise FileNotFoundError(
            'metadata/image_concepts_binary.csv not found. '
            'Run prepare_funnybirds_metadata.py first.')
    wide = pd.read_csv(concepts_csv)
    concept_cols = [c for c in wide.columns if c != 'image_id']
    long = wide.melt(id_vars='image_id', value_vars=concept_cols,
                     var_name='attr_name', value_name='is_present')
    long['attr_id']    = long.groupby('attr_name', sort=False).ngroup()
    long['is_present'] = long['is_present'].astype(int)
    long['certainty']  = 1
    return long[['image_id', 'attr_id', 'attr_name', 'is_present', 'certainty']]


def load_attr_maps(fb_root: Path):
    """
    Load concept names from metadata/concepts.csv.
    Returns attr_id_to_name, attr_name_to_id.
    Mirrors load_attr_maps(attr_txt) in recall.ipynb.
    """
    concepts_csv = fb_root / 'metadata' / 'concepts.csv'
    if not concepts_csv.exists():
        raise FileNotFoundError(
            'metadata/concepts.csv not found. Run prepare_funnybirds_metadata.py first.')
    df = pd.read_csv(concepts_csv)
    id2name = dict(zip(df['concept_id'], df['concept_name']))
    name2id = dict(zip(df['concept_name'], df['concept_id']))
    return id2name, name2id

In [4]:
meta                           = load_meta(FB)
img_attr_long                  = load_image_attr_labels_robust(FB)
attr_id_to_name, attr_name_to_id = load_attr_maps(FB)

attr_df = pd.DataFrame({
    'attr_name': list(attr_name_to_id.keys()),
    'attr_id':   list(attr_name_to_id.values()),
})

id2name = load_species_maps(FB)
spname  = lambda sid: id2name.get(int(sid), f'funnybird_{int(sid):02d}')

print(f'meta: {len(meta)} images '
      f'({meta["is_train"].sum()} train, {(meta["is_train"]==0).sum()} test)')
print(f'img_attr_long: {len(img_attr_long)} rows, '
      f'{img_attr_long["attr_name"].nunique()} concepts')
print(f'Concepts: {list(attr_name_to_id.keys())}')

meta: 50500 images (50000 train, 500 test)
img_attr_long: 1313000 rows, 26 concepts
Concepts: ['beak_0', 'beak_1', 'beak_2', 'beak_3', 'eye_0', 'eye_1', 'eye_2', 'wing_0', 'wing_1', 'wing_2', 'wing_3', 'wing_4', 'wing_5', 'foot_0', 'foot_1', 'foot_2', 'foot_3', 'tail_0', 'tail_1', 'tail_2', 'tail_3', 'tail_4', 'tail_5', 'tail_6', 'tail_7', 'tail_8']


In [5]:
def build_attr_labeled_df(
    meta: pd.DataFrame,
    img_attr_long: pd.DataFrame,
    attr_id: int,
    min_certainty: int = 1,
) -> pd.DataFrame:
    """
    Returns dataframe with:
        image_id, species_id, species_name, is_train, y, certainty
    Only keeps annotations with certainty >= min_certainty.
    Mirrors build_attr_labeled_df in recall.ipynb verbatim.
    """
    sub = img_attr_long[img_attr_long['attr_id'] == int(attr_id)].copy()
    sub = sub[sub['certainty'] >= int(min_certainty)].copy()
    out = meta.merge(sub[['image_id', 'is_present', 'certainty']], on='image_id', how='inner')
    out = out.rename(columns={'is_present': 'y'})
    out['y'] = out['y'].astype(int)
    return out[['image_id', 'species_id', 'species_name', 'is_train', 'y', 'certainty']]


print('Defined: build_attr_labeled_df')

# sanity check on one concept
attr_name = attr_df['attr_name'].iloc[0]
aid       = attr_name_to_id[attr_name]
lab       = build_attr_labeled_df(meta, img_attr_long, aid, min_certainty=1)
print(f'Attribute: {attr_name}  rows labeled: {len(lab)}  pos rate: {lab.y.mean():.4f}')
lab.head()

Defined: build_attr_labeled_df
Attribute: beak_0  rows labeled: 50500  pos rate: 0.3174


,image_id,species_id,species_name,is_train,y,certainty
0,0,0,funnybird_00,1,0,1
1,1,0,funnybird_00,1,0,1
2,2,0,funnybird_00,1,0,1
3,3,0,funnybird_00,1,0,1
4,4,0,funnybird_00,1,0,1


In [6]:
def safe_torch_load(path: Path):
    try:
        return torch.load(path, map_location='cpu', weights_only=True)
    except TypeError:
        return torch.load(path, map_location='cpu')


def load_features(feat_dir: Path, layer: str, split: str) -> torch.Tensor:
    p = feat_dir / f'{layer}_{split}.pt'
    assert p.exists(), f'Missing: {p}'
    X = safe_torch_load(p)
    if not isinstance(X, torch.Tensor):
        X = torch.tensor(X)
    return X.float()


def to_1d_int_array(x):
    if isinstance(x, torch.Tensor):
        x = x.detach().cpu().numpy()
    x = np.array(x).reshape(-1)
    return x.astype(int)


def infer_kind(arr):
    if arr.max() <= 200 and arr.min() >= 0:
        return 'species_id_like'
    if arr.max() > 200:
        return 'image_id_like'
    return 'unknown'


def load_split_order(feat_dir, split):
    p = feat_dir / f'labels_{split}.pt'
    assert p.exists(), f'Missing: {p}'
    t = torch.load(p, map_location='cpu', weights_only=True)
    assert isinstance(t, dict), f'Expected dict in {p}, got {type(t)}'
    assert 'image_ids' in t, f'{p} missing image_ids key; has {list(t.keys())}'
    ids  = to_1d_int_array(t['image_ids'])
    kind = infer_kind(ids)
    return kind, ids

In [7]:
LAYER = 'layer4.0'

base_kind_tr, base_ids_tr = load_split_order(BASE_FEATS, 'train') if BASE_FEATS.exists() else (None, None)
base_kind_te, base_ids_te = load_split_order(BASE_FEATS, 'test')  if BASE_FEATS.exists() else (None, None)
cbm_kind_tr,  cbm_ids_tr  = load_split_order(CBM_FEATS,  'train') if CBM_FEATS.exists()  else (None, None)
cbm_kind_te,  cbm_ids_te  = load_split_order(CBM_FEATS,  'test')  if CBM_FEATS.exists()  else (None, None)

print('BASE_FEATS available:', BASE_FEATS.exists())
print('CBM_FEATS  available:', CBM_FEATS.exists())

if BASE_FEATS.exists():
    print(f'  Baseline train: {base_kind_tr}  id range: ({base_ids_tr.min()}, {base_ids_tr.max()})')
    print(f'  Baseline test:  {base_kind_te}  id range: ({base_ids_te.min()}, {base_ids_te.max()})')
if CBM_FEATS.exists():
    print(f'  CBM train: {cbm_kind_tr}  id range: ({cbm_ids_tr.min()}, {cbm_ids_tr.max()})')
    print(f'  CBM test:  {cbm_kind_te}  id range: ({cbm_ids_te.min()}, {cbm_ids_te.max()})')

BASE_FEATS available: True
CBM_FEATS  available: True
  Baseline train: image_id_like  id range: (0, 49999)
  Baseline test:  image_id_like  id range: (50000, 50499)
  CBM train: image_id_like  id range: (0, 49999)
  CBM test:  image_id_like  id range: (50000, 50499)


In [8]:
def align_features_and_labels(
    X_split: torch.Tensor,
    image_ids_in_feature_order: np.ndarray,
    labeled_df_split: pd.DataFrame,
):
    """
    Aligns features (in feature-row order) to attribute labels by image_id.
    Mirrors align_features_and_labels in recall.ipynb verbatim.

    Inputs:
        X_split                    : feature tensor [N, D]
        image_ids_in_feature_order : length-N int array, image_id per row of X_split
        labeled_df_split           : dataframe with [image_id, y, species_id, species_name]
    Returns:
        X_aligned  : features for images that have labels
        df_aligned : same rows, same order, includes y and species info
    """
    labeled  = labeled_df_split.set_index('image_id')[['y', 'species_id', 'species_name']]
    keep_idx = []
    rows     = []
    for i, img_id in enumerate(image_ids_in_feature_order):
        img_id = int(img_id)
        if img_id in labeled.index:
            keep_idx.append(i)
            y, sid, sname = labeled.loc[img_id]
            rows.append((img_id, int(sid), str(sname), int(y)))
    X_aligned  = X_split[keep_idx]
    df_aligned = pd.DataFrame(rows, columns=['image_id', 'species_id', 'species_name', 'y'])
    return X_aligned, df_aligned

In [9]:
class LinearProbe(nn.Module):
    def __init__(self, d: int):
        super().__init__()
        self.lin = nn.Linear(d, 1)
    def forward(self, x):
        return self.lin(x).squeeze(-1)


def train_probe(Xtr: torch.Tensor, ytr: np.ndarray,
                seed=0, lr=1e-2, wd=1e-4, epochs=25, batch=512):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    Xtr    = Xtr.to(device)
    ytr_t  = torch.tensor(ytr, dtype=torch.float32, device=device)
    probe  = LinearProbe(Xtr.shape[1]).to(device)
    opt    = torch.optim.AdamW(probe.parameters(), lr=lr, weight_decay=wd)
    pos    = float(ytr_t.mean().item())
    pw     = torch.tensor([(1 - pos) / pos], device=device) if 0 < pos < 1 \
             else torch.tensor([1.0], device=device)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pw)

    n = Xtr.shape[0]
    for _ in range(epochs):
        perm = torch.randperm(n, device=device)
        for i in range(0, n, batch):
            idx    = perm[i:i+batch]
            logits = probe(Xtr[idx])
            loss   = loss_fn(logits, ytr_t[idx])
            opt.zero_grad(); loss.backward(); opt.step()
    return probe


@torch.no_grad()
def predict_probs(probe: nn.Module, X: torch.Tensor, batch=4096) -> np.ndarray:
    probe.eval()
    probs = []
    for i in range(0, X.shape[0], batch):
        xb = X[i:i+batch].to(device)
        probs.append(torch.sigmoid(probe(xb)).detach().cpu())
    return torch.cat(probs, dim=0).numpy()


print('Defined: LinearProbe  train_probe  predict_probs')

Defined: LinearProbe  train_probe  predict_probs


In [10]:
def make_candidate_pairs(df_test: pd.DataFrame, min_each=10, max_pairs=200, seed=0):
    """
    CUB-style: requires species with BOTH min_each positives AND min_each negatives.
    Always returns [] for FunnyBirds (concepts are 100% present or absent per species).
    Retained so run_one_attribute can use it as a gate: if it returns [] we fall through
    to make_candidate_pairs_fb.
    """
    g    = df_test.groupby('species_id')['y'].agg(['count', 'sum']).rename(columns={'sum': 'pos'})
    g['neg'] = g['count'] - g['pos']
    ok   = g[(g['pos'] >= min_each) & (g['neg'] >= min_each)]
    sids = ok.index.to_list()
    rng  = np.random.default_rng(seed)
    pairs = []
    if len(sids) < 2:
        return pairs
    for _ in range(max_pairs * 10):
        a, b = rng.choice(sids, size=2, replace=False)
        mpos = int(min(ok.loc[a, 'pos'], ok.loc[b, 'pos']))
        mneg = int(min(ok.loc[a, 'neg'], ok.loc[b, 'neg']))
        if mpos >= min_each and mneg >= min_each:
            pairs.append((int(a), int(b), mpos, mneg))
        if len(pairs) >= max_pairs:
            break
    return pairs


def make_candidate_pairs_fb(
    df_test: pd.DataFrame,
    min_pos: int = 3,
    max_pairs: int = 200,
    seed: int = 0,
):
    """
    FunnyBirds variant: enumerate all pairs of all-positive species
    (prevalence >= 0.9 AND n_pos >= min_pos).
    Returns list of (sid_A, sid_B, mpos).
    """
    from itertools import combinations as _combinations
    g    = df_test.groupby('species_id')['y'].agg(['count', 'sum']).rename(columns={'sum': 'pos'})
    g['prev'] = g['pos'] / g['count']
    ok   = g[(g['pos'] >= min_pos) & (g['prev'] >= 0.9)]
    sids = ok.index.tolist()
    rng  = np.random.default_rng(seed)
    rng.shuffle(sids)
    pairs = []
    for a, b in _combinations(sids, 2):
        mpos = int(min(ok.loc[a, 'pos'], ok.loc[b, 'pos']))
        pairs.append((int(a), int(b), mpos))
        if len(pairs) >= max_pairs:
            break
    return pairs

In [11]:
def fb_pair_eval(
    df_test: pd.DataFrame,
    probs: np.ndarray,
    sid_A: int,
    sid_B: int,
    mpos: int,
    seed: int = 0,
    thr: float = 0.5,
):
    """
    FunnyBirds matched-pair evaluation for all-positive species.
    Samples mpos images from each all-positive species (with replacement if needed)
    and computes recall = fraction classified positive at thr.
    gap = |recall_A - recall_B|.

    This is the FunnyBirds port of CUB's matched_pair_eval:
    CUB controls prevalence by subsampling pos+neg; here we only have positives,
    so we subsample mpos images from each positive species to equalize sample size.
    """
    df        = df_test.copy()
    df['prob'] = probs
    A = df[(df.species_id == sid_A) & (df.y == 1)]
    B = df[(df.species_id == sid_B) & (df.y == 1)]

    n_A = min(mpos, len(A))
    n_B = min(mpos, len(B))
    A_s = A.sample(n_A, random_state=seed, replace=(n_A < mpos))
    B_s = B.sample(n_B, random_state=seed, replace=(n_B < mpos))

    def recall_pos(d):
        pred = (d.prob.values >= thr).astype(int)
        return float(pred.mean()) if len(d) > 0 else np.nan

    recA = recall_pos(A_s)
    recB = recall_pos(B_s)
    return {
        'sid_A':     sid_A,
        'sid_B':     sid_B,
        'species_A': spname(sid_A),
        'species_B': spname(sid_B),
        'npos':      int(mpos),
        'nneg':      0,
        'recall_A':  float(recA),
        'recall_B':  float(recB),
        'gap':       float(abs(recA - recB)),
    }

In [12]:
def bootstrap_ci(x, alpha=0.05):
    """
    Percentile bootstrap CI for a 1D array x.
    Returns (lo, hi). If empty or all-nan returns (nan, nan).
    Mirrors recall.ipynb verbatim.
    """
    x = np.asarray(x, dtype=float)
    x = x[~np.isnan(x)]
    if x.size == 0:
        return (np.nan, np.nan)
    return (float(np.quantile(x, alpha / 2)),
            float(np.quantile(x, 1 - alpha / 2)))


def bootstrap_p_value(values, null=0.0):
    """
    Two-sided bootstrap p-value for H0: E[value] == null.
    p = 2 * min(P(value <= null), P(value >= null)).
    Mirrors recall.ipynb verbatim.
    """
    v = np.asarray(values, dtype=float)
    v = v[~np.isnan(v)]
    if v.size == 0:
        return np.nan
    p_lo = np.mean(v <= null)
    p_hi = np.mean(v >= null)
    return float(2.0 * min(p_lo, p_hi))


def safe_div(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    out = np.full_like(a, np.nan, dtype=float)
    m = b != 0
    out[m] = a[m] / b[m]
    return out


print('Defined: bootstrap_ci  bootstrap_p_value  safe_div')

Defined: bootstrap_ci  bootstrap_p_value  safe_div


In [13]:
def fb_bootstrap_summary(
    df_te: pd.DataFrame,
    probs: np.ndarray,
    pairs,
    *,
    thr: float = 0.5,
    B: int = 300,
):
    """
    FunnyBirds port of CUB's matched_pair_bootstrap_summary.
    Accepts 3-tuple pairs (sid_A, sid_B, mpos) from make_candidate_pairs_fb.
    Output column schema identical to CUB's matched_pair_bootstrap_summary
    so all downstream code (summarize_by_attr, top_pairs, plots) is unchanged.
    """
    matched_cols = [
        'sid_A', 'sid_B', 'species_A', 'species_B',
        'npos', 'nneg', 'recall_A', 'recall_B', 'gap', 'boot_id'
    ]
    summary_cols = [
        'sid_A', 'sid_B', 'species_A', 'species_B',
        'npos', 'nneg',
        'gap_mean', 'gap_std', 'gap_ci_lo', 'gap_ci_hi', 'gap_p',
        'gap_ci_width', 'gap_snr', 'gap_norm', 'n_runs'
    ]

    if pairs is None or len(pairs) == 0:
        return pd.DataFrame(columns=matched_cols), pd.DataFrame(columns=summary_cols)

    rows = []
    for (a, b, mpos) in pairs:
        for boot_id in range(B):
            r = fb_pair_eval(df_te, probs,
                             sid_A=int(a), sid_B=int(b),
                             mpos=int(mpos), seed=int(boot_id), thr=float(thr))
            r['boot_id'] = int(boot_id)
            rows.append(r)

    res_long = pd.DataFrame(rows).dropna(subset=['gap'])
    if res_long.empty:
        return res_long, pd.DataFrame(columns=summary_cols)

    def _ci_lo(x): return bootstrap_ci(x)[0]
    def _ci_hi(x): return bootstrap_ci(x)[1]

    pair_summary = (
        res_long.groupby(['sid_A', 'sid_B', 'species_A', 'species_B'], as_index=False)
        .agg(
            npos      = ('npos',  'min'),
            nneg      = ('nneg',  'min'),
            gap_mean  = ('gap',   'mean'),
            gap_std   = ('gap',   'std'),
            gap_ci_lo = ('gap',   _ci_lo),
            gap_ci_hi = ('gap',   _ci_hi),
            gap_p     = ('gap',   bootstrap_p_value),
            n_runs    = ('gap',   'size'),
        )
    )
    EPS = 1e-12
    pair_summary['gap_ci_width'] = pair_summary['gap_ci_hi'] - pair_summary['gap_ci_lo']
    pair_summary['gap_snr']  = pair_summary['gap_mean'] / (pair_summary['gap_std'].fillna(0.0) + EPS)
    pair_summary['gap_norm'] = pair_summary['gap_mean']
    pair_summary = pair_summary.sort_values('gap_mean', ascending=False).reset_index(drop=True)
    return res_long, pair_summary


print('Defined: fb_bootstrap_summary')

Defined: fb_bootstrap_summary


In [14]:
def species_recall_prevalence_table(df_te: pd.DataFrame, probs: np.ndarray, thr=0.5) -> pd.DataFrame:
    """
    Per-species table on the TEST set:
    n, n_pos, n_neg, prevalence, tp, recall, precision.
    Mirrors recall.ipynb verbatim.
    """
    df = df_te[['species_id', 'species_name', 'y']].copy()
    df['prob'] = np.asarray(probs, dtype=float)
    df['pred'] = (df['prob'] >= thr).astype(int)

    g = (df.groupby(['species_id', 'species_name'], as_index=False)
           .agg(n=('y', 'size'), n_pos=('y', 'sum'), n_pred_pos=('pred', 'sum')))
    g['n_neg']      = g['n'] - g['n_pos']
    g['prevalence'] = g['n_pos'] / g['n']

    tp = (df[df['y'] == 1]
            .groupby(['species_id', 'species_name'])['pred']
            .sum().reset_index(name='tp'))
    out = g.merge(tp, on=['species_id', 'species_name'], how='left')
    out['tp']        = out['tp'].fillna(0).astype(int)
    out['recall']    = np.where(out['n_pos'] > 0, out['tp'] / out['n_pos'], np.nan)
    out['precision'] = np.where(out['n_pred_pos'] > 0, out['tp'] / out['n_pred_pos'], np.nan)
    return out.sort_values('n', ascending=False).reset_index(drop=True)


def add_species_bootstrap_ci(
    df_te: pd.DataFrame,
    probs: np.ndarray,
    thr=0.5,
    B=300,
    min_pos_for_ci=1,
) -> pd.DataFrame:
    """
    Adds bootstrap CIs for per-species recall by resampling each species'
    test images with replacement.
    New columns: recall_ci_lo, recall_ci_hi, recall_bs_mean, recall_ci_width.
    Mirrors recall.ipynb verbatim.
    """
    df = df_te[['species_id', 'species_name', 'y']].copy()
    df['prob'] = np.asarray(probs, dtype=float)
    rows = []
    rng  = np.random.default_rng(0)

    for (sid, sname), d in df.groupby(['species_id', 'species_name']):
        d     = d.reset_index(drop=True)
        n     = len(d)
        n_pos = int(d['y'].sum())

        if n_pos < min_pos_for_ci:
            rows.append(dict(species_id=int(sid), species_name=str(sname),
                             recall_bs_mean=np.nan, recall_ci_lo=np.nan,
                             recall_ci_hi=np.nan, recall_ci_width=np.nan, B=int(B)))
            continue

        vals = []
        for _ in range(B):
            idx = rng.integers(0, n, size=n)
            s   = d.iloc[idx]
            pos = s[s['y'] == 1]
            if len(pos) == 0:
                vals.append(np.nan); continue
            vals.append(float((pos['prob'].to_numpy() >= thr).mean()))

        vals = np.asarray(vals, dtype=float)
        lo, hi = bootstrap_ci(vals, alpha=0.05)
        rows.append(dict(species_id=int(sid), species_name=str(sname),
                         recall_bs_mean=float(np.nanmean(vals)),
                         recall_ci_lo=lo, recall_ci_hi=hi,
                         recall_ci_width=(hi - lo) if (np.isfinite(lo) and np.isfinite(hi)) else np.nan,
                         B=int(B)))

    ci_df = pd.DataFrame(rows)
    base  = species_recall_prevalence_table(df_te, probs, thr=thr)
    return base.merge(ci_df, on=['species_id', 'species_name'], how='left')


print('Defined: species_recall_prevalence_table  add_species_bootstrap_ci')

Defined: species_recall_prevalence_table  add_species_bootstrap_ci


In [15]:
def run_one_attribute(
    attr_name: str,
    feat_dir: Path,
    split_order_kind_train: str,
    split_order_train: np.ndarray,
    split_order_kind_test: str,
    split_order_test: np.ndarray,
    layer: str,
    *,
    min_certainty: int = 1,
    thr: float = 0.5,
    epochs: int = 25,
    min_each: int = 10,
    n_pairs: int = 200,
    B_gap: int = 100,
    B_species: int = 100,
    use_balanced_acc: bool = False,
):
    """
    Full single-concept pipeline.
    1) Build labeled train/test sets (certainty filtering)
    2) Load train/test features for chosen layer
    3) Align features to labels by image_id
    4) Train linear probe
    5) Predict probabilities on test set
    6) Per-species recall table + bootstrap CI
    7) Matched pairs: try CUB-style first (always [] for FunnyBirds),
       then fall through to fb_bootstrap_summary with all-positive pairs.
    """
    assert split_order_kind_train == 'image_id_like' and split_order_kind_test == 'image_id_like', (
        'Cannot align features to attribute labels: labels_{split}.pt is not image_id-like. '
        'Check your feature extractor output format.')

    aid       = attr_name_to_id[attr_name]
    lab       = build_attr_labeled_df(meta, img_attr_long, aid, min_certainty=min_certainty)
    lab_train = lab[lab['is_train'] == 1].copy()
    lab_test  = lab[lab['is_train'] == 0].copy()

    Xtr_all = load_features(feat_dir, layer, 'train')
    Xte_all = load_features(feat_dir, layer, 'test')

    Xtr, df_tr = align_features_and_labels(Xtr_all, split_order_train, lab_train)
    Xte, df_te = align_features_and_labels(Xte_all, split_order_test,  lab_test)

    ytr = df_tr['y'].astype(int).to_numpy()
    yte = df_te['y'].astype(int).to_numpy()

    probe = train_probe(Xtr, ytr, seed=0, epochs=epochs)
    probs = predict_probs(probe, Xte)

    # Per-species table
    species_table = add_species_bootstrap_ci(df_te, probs, thr=thr, B=B_species)

    # Overall accuracy (plain or balanced)
    if len(yte) == 0:
        test_acc = np.nan
    elif use_balanced_acc:
        pred_bin = (probs >= thr).astype(int)
        tp  = int(((pred_bin == 1) & (yte == 1)).sum())
        tn  = int(((pred_bin == 0) & (yte == 0)).sum())
        fp  = int(((pred_bin == 1) & (yte == 0)).sum())
        fn  = int(((pred_bin == 0) & (yte == 1)).sum())
        tpr = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        tnr = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        test_acc = 0.5 * (tpr + tnr)
    else:
        test_acc = float(((probs >= thr).astype(int) == yte).mean())

    # Matched pairs
    if n_pairs is None or int(n_pairs) <= 0:
        res_long = pd.DataFrame(); pair_summary = pd.DataFrame(); pairs = []
    else:
        # CUB gate: try standard pairs (always [] for FunnyBirds)
        pairs_cub = make_candidate_pairs(df_te, min_each=min_each, max_pairs=n_pairs, seed=0)
        if len(pairs_cub) > 0:
            # Should never reach here for FunnyBirds, but kept for correctness
            raise RuntimeError('Unexpected CUB-style pairs found in FunnyBirds data')
        else:
            # FunnyBirds path: pair all-positive species
            pairs = make_candidate_pairs_fb(
                df_te, min_pos=max(1, min_each // 5), max_pairs=n_pairs, seed=0)
            res_long, pair_summary = fb_bootstrap_summary(
                df_te, probs, pairs, thr=thr, B=B_gap)

    mean_gap = float(pair_summary['gap_mean'].mean()) \
               if (pair_summary is not None and len(pair_summary)) else np.nan
    p90_gap  = float(pair_summary['gap_mean'].quantile(0.9)) \
               if (pair_summary is not None and len(pair_summary)) else np.nan

    info = {
        'attr':             attr_name,
        'layer':            layer,
        'n_train':          int(len(df_tr)),
        'n_test':           int(len(df_te)),
        'train_pos_rate':   float(ytr.mean()) if len(ytr) else np.nan,
        'test_pos_rate':    float(yte.mean()) if len(yte) else np.nan,
        'test_acc':         float(test_acc),
        'thr':              float(thr),
        'epochs':           int(epochs),
        'n_pairs':          int(len(pairs)),
        'B_gap':            int(B_gap),
        'B_species':        int(B_species),
        'mean_gap':         mean_gap,
        'p90_gap':          p90_gap,
        'use_balanced_acc': bool(use_balanced_acc),
    }
    return info, res_long, pair_summary, df_te, species_table


print('Defined: run_one_attribute')

Defined: run_one_attribute


In [16]:
def screen_attributes_for_species_variation(
    candidate_attrs,
    feat_dir: Path,
    kind_tr: str, ids_tr: np.ndarray,
    kind_te: str, ids_te: np.ndarray,
    layer: str,
    *,
    min_certainty: int = 1,
    thr: float = 0.5,
    min_pos_per_species: int = 10,
    min_species_with_pos: int = 15,
    min_overall_prev: float = 0.05,
    max_overall_prev: float = 0.95,
    epochs: int = 8,
    max_attrs=None,
    verbose_every: int = 50,
    keep_error_examples: int = 5,
    B_species: int = 200,
):
    rows   = []
    errors = []
    stats  = dict(tried=0, success=0,
                  filtered_too_few_species_pos=0,
                  filtered_prev_out_of_range=0,
                  filtered_no_recall_vals=0,
                  errored=0)

    cand = list(candidate_attrs)
    if max_attrs is not None:
        cand = cand[:max_attrs]

    for i, attr in enumerate(cand):
        stats['tried'] += 1
        try:
            info, _, _, _, species_table = run_one_attribute(
                attr, feat_dir,
                kind_tr, ids_tr, kind_te, ids_te,
                layer=layer,
                min_certainty=min_certainty, thr=thr,
                epochs=epochs,
                min_each=10, n_pairs=0,   # skip matched pairs during screening
                B_species=B_species,
            )
            st            = species_table.copy()
            overall_prev  = float(st['n_pos'].sum() / st['n'].sum()) if st['n'].sum() > 0 else np.nan
            st_pos        = st[st['n_pos'] >= min_pos_per_species].copy()
            n_species_pos = int(len(st_pos))

            if n_species_pos < min_species_with_pos:
                stats['filtered_too_few_species_pos'] += 1; continue
            if not (min_overall_prev <= overall_prev <= max_overall_prev):
                stats['filtered_prev_out_of_range'] += 1; continue

            recall_vals = st_pos['recall'].dropna().to_numpy()
            if recall_vals.size == 0:
                stats['filtered_no_recall_vals'] += 1; continue

            stats['success'] += 1
            rows.append({
                'attr':           attr,
                'overall_prev':   overall_prev,
                'n_species_pos':  n_species_pos,
                'recall_std':     float(np.std(recall_vals)),
                'recall_range':   float(np.max(recall_vals) - np.min(recall_vals)),
                'recall_p90_p10': float(np.quantile(recall_vals, 0.9) - np.quantile(recall_vals, 0.1)),
                'test_acc':       float(info['test_acc']),
                'n_test':         int(info['n_test']),
            })
            if verbose_every and ((i + 1) % verbose_every == 0):
                print(f'[{i+1}/{len(cand)}] ok: {attr} '
                      f'prev={overall_prev:.3f} n_species_pos={n_species_pos}')
        except Exception as e:
            stats['errored'] += 1
            if len(errors) < keep_error_examples:
                errors.append((attr, repr(e)))
            continue

    screen_df = pd.DataFrame(rows)
    print('\n--- Screening summary ---')
    for k, v in stats.items():
        print(f'  {k}: {v}')
    if errors:
        print('\nExample errors (first few):')
        for a, msg in errors:
            print(f'  {a} -> {msg}')
    if screen_df.empty:
        print('\nNo concepts passed filters. Likely causes:')
        print('  - run_one_attribute erroring for most concepts (see errors above)')
        print('  - filters too strict for this concept distribution')
        return screen_df

    # Sort keys identical to recall.ipynb
    screen_df = screen_df.sort_values(
        ['recall_p90_p10', 'recall_range', 'recall_std'], ascending=False
    ).reset_index(drop=True)
    return screen_df


print('Defined: screen_attributes_for_species_variation')

Defined: screen_attributes_for_species_variation


In [17]:
# FunnyBirds screening thresholds:
#   - 500 test images, 10 per class → each positive species has all 10 test images positive
#   - tail has 9 variants → ~5-6 species per variant → min_species_with_pos must be ≤5
#   - concepts are perfectly balanced → no strict prevalence filter needed
CANDIDATE_ATTRS = attr_df['attr_name'].tolist()

_screen_feat = CBM_FEATS if CBM_FEATS.exists() else BASE_FEATS
_sk_tr = cbm_kind_tr if CBM_FEATS.exists() else base_kind_tr
_si_tr = cbm_ids_tr  if CBM_FEATS.exists() else base_ids_tr
_sk_te = cbm_kind_te if CBM_FEATS.exists() else base_kind_te
_si_te = cbm_ids_te  if CBM_FEATS.exists() else base_ids_te

screen_df = screen_attributes_for_species_variation(
    CANDIDATE_ATTRS,
    feat_dir              = _screen_feat,
    kind_tr=_sk_tr, ids_tr=_si_tr,
    kind_te=_sk_te, ids_te=_si_te,
    layer                 = LAYER,
    min_certainty         = 1,
    thr                   = 0.5,
    min_pos_per_species   = 3,
    min_species_with_pos  = 4,
    min_overall_prev      = 0.01,
    max_overall_prev      = 0.99,
    epochs                = 8,
    max_attrs             = 200,
    verbose_every         = 5,
    B_species             = 200,
)

if not screen_df.empty:
    TOP_K     = 26   # FunnyBirds has ≤26 concepts; use all that pass
    ATTR_LIST = screen_df['attr'].head(TOP_K).tolist()
    print(f'\nATTR_LIST ({len(ATTR_LIST)} concepts):')
    for a in ATTR_LIST:
        print(f'  {a}')
    display(screen_df.head(30))
else:
    ATTR_LIST = CANDIDATE_ATTRS
    print('[warn] No concepts passed screening. Using full ATTR_LIST.')

# attr_to_spread: used for normalization in plots (mirrors recall.ipynb)
attr_to_spread = screen_df.set_index('attr')['recall_p90_p10'].to_dict() \
                 if not screen_df.empty else {}

[5/26] ok: eye_0 prev=0.300 n_species_pos=15
[10/26] ok: wing_2 prev=0.220 n_species_pos=11
[15/26] ok: foot_1 prev=0.200 n_species_pos=10
[20/26] ok: tail_2 prev=0.140 n_species_pos=7

--- Screening summary ---
  tried: 26
  success: 25
  filtered_too_few_species_pos: 1
  filtered_prev_out_of_range: 0
  filtered_no_recall_vals: 0
  errored: 0

ATTR_LIST (25 concepts):
  tail_5
  tail_1
  tail_8
  eye_0
  beak_3
  beak_0
  foot_3
  tail_4
  beak_2
  tail_3
  eye_2
  beak_1
  wing_4
  eye_1
  wing_0
  wing_1
  wing_2
  wing_3
  wing_5
  foot_0
  foot_1
  foot_2
  tail_0
  tail_2
  tail_6


,attr,overall_prev,n_species_pos,recall_std,recall_range,recall_p90_p10,test_acc,n_test
0,tail_5,0.16,8,0.131696,0.4,0.19,0.946,500
1,tail_1,0.10,5,0.074833,0.2,0.16,0.948,500
2,tail_8,0.16,8,0.069597,0.2,0.13,0.950,500
3,eye_0,0.30,15,0.061824,0.2,0.10,0.984,500
4,beak_3,0.26,13,0.060569,0.2,0.10,0.982,500
5,beak_0,0.42,21,0.045175,0.1,0.10,0.978,500
6,foot_3,0.26,13,0.036080,0.1,0.08,0.994,500
7,tail_4,0.08,4,0.043301,0.1,0.07,0.934,500
8,beak_2,0.12,6,0.037268,0.1,0.05,0.988,500
9,tail_3,0.14,7,0.034993,0.1,0.04,0.978,500


In [18]:
def run_many(
    attr_list,
    model_name,
    feat_dir,
    kind_tr, ids_tr,
    kind_te, ids_te,
    *,
    layer,
    min_certainty=1,
    thr=0.5,
    epochs=25,
    min_each=10,
    n_pairs=200,
    B_gap=100,
    B_species=100,
    use_balanced_acc=False,
):
    """
    Runs run_one_attribute over a list of concepts. Collects:
    - info_df    : one row per concept (headline metrics: acc, mean gap, etc.)
    - pairs_df   : per-(concept, species pair) summary (gap_mean, CI, p, ...)
    - species_df : per-(concept, species) table (prevalence, recall, recall CI)
    Mirrors recall.ipynb verbatim except use_balanced_acc param added.
    """
    all_info      = []
    all_pair_summ = []
    all_species   = []

    for attr in attr_list:
        info, _, pair_summ, _, species_table = run_one_attribute(
            attr, feat_dir,
            kind_tr, ids_tr, kind_te, ids_te,
            layer            = layer,
            min_certainty    = min_certainty,
            thr              = thr,
            epochs           = epochs,
            min_each         = min_each,
            n_pairs          = n_pairs,
            B_gap            = B_gap,
            B_species        = B_species,
            use_balanced_acc = use_balanced_acc,
        )
        info          = dict(info)
        info['model'] = model_name
        all_info.append(info)

        if pair_summ is not None and len(pair_summ):
            ps          = pair_summ.copy()
            ps['attr']  = attr
            ps['model'] = model_name
            all_pair_summ.append(ps)

        st          = species_table.copy()
        st['attr']  = attr
        st['model'] = model_name
        all_species.append(st)

        acc_label = 'bal_acc' if use_balanced_acc else 'test_acc'
        print(model_name, attr,
              f'{acc_label}=', round(info['test_acc'], 4),
              'mean_gap=',     round(info['mean_gap'], 4))

    info_df    = pd.DataFrame(all_info)
    pairs_df   = pd.concat(all_pair_summ, ignore_index=True) if all_pair_summ else pd.DataFrame()
    species_df = pd.concat(all_species,   ignore_index=True) if all_species   else pd.DataFrame()
    return info_df, pairs_df, species_df


print('Defined: run_many')

Defined: run_many


In [19]:
results = {}

if BASE_FEATS.exists():
    print('=== Running baseline ===')
    info_base, pairs_base, species_base = run_many(
        ATTR_LIST, 'baseline_fb', BASE_FEATS,
        base_kind_tr, base_ids_tr,
        base_kind_te, base_ids_te,
        layer=LAYER, thr=0.5, n_pairs=200, B_gap=100, B_species=100,
    )
    results['baseline'] = (info_base, pairs_base, species_base)
    species_base.to_csv('fb_baseline_species.csv', index=False)
    print(f'\nBaseline done: mean_gap={info_base.mean_gap.mean():.4f}  '
          f'mean_acc={info_base.test_acc.mean():.4f}')
else:
    print('[warn] Baseline features not found.')

if CBM_FEATS.exists():
    print('\n=== Running CBM ===')
    info_cbm, pairs_cbm, species_cbm = run_many(
        ATTR_LIST, 'cbm_fb', CBM_FEATS,
        cbm_kind_tr, cbm_ids_tr,
        cbm_kind_te, cbm_ids_te,
        layer=LAYER, thr=0.5, n_pairs=200, B_gap=100, B_species=100,
    )
    results['cbm'] = (info_cbm, pairs_cbm, species_cbm)
    species_cbm.to_csv('fb_cbm_species.csv', index=False)
    print(f'\nCBM done: mean_gap={info_cbm.mean_gap.mean():.4f}  '
          f'mean_acc={info_cbm.test_acc.mean():.4f}')
else:
    print('[warn] CBM features not found.')

# Quick sanity check: compare concept-level summaries
if 'baseline' in results and 'cbm' in results:
    info_base, cbm_info = results['baseline'][0], results['cbm'][0]
    info_base, cbm_info

=== Running baseline ===
baseline_fb tail_5 test_acc= 0.938 mean_gap= 0.05
baseline_fb tail_1 test_acc= 0.958 mean_gap= 0.04
baseline_fb tail_8 test_acc= 0.972 mean_gap= 0.0679
baseline_fb eye_0 test_acc= 0.988 mean_gap= 0.0248
baseline_fb beak_3 test_acc= 0.992 mean_gap= 0.0282
baseline_fb beak_0 test_acc= 0.986 mean_gap= 0.038
baseline_fb foot_3 test_acc= 0.994 mean_gap= 0.0282
baseline_fb tail_4 test_acc= 0.958 mean_gap= 0.05
baseline_fb beak_2 test_acc= 0.992 mean_gap= 0.0
baseline_fb tail_3 test_acc= 0.986 mean_gap= 0.0286
baseline_fb eye_2 test_acc= 0.996 mean_gap= 0.0
baseline_fb beak_1 test_acc= 1.0 mean_gap= 0.0
baseline_fb wing_4 test_acc= 0.996 mean_gap= 0.02
baseline_fb eye_1 test_acc= 0.99 mean_gap= 0.0325
baseline_fb wing_0 test_acc= 0.998 mean_gap= 0.0
baseline_fb wing_1 test_acc= 0.998 mean_gap= 0.0222
baseline_fb wing_2 test_acc= 0.998 mean_gap= 0.0
baseline_fb wing_3 test_acc= 0.998 mean_gap= 0.04
baseline_fb wing_5 test_acc= 0.998 mean_gap= 0.0
baseline_fb foot_0 tes

In [20]:
def add_gap_interpretability_cols(pair_df: pd.DataFrame) -> pd.DataFrame:
    """
    Adds interpretability columns to a pair_summary dataframe.
    Mirrors recall.ipynb verbatim.

    - gap_snr      : gap_mean / gap_std  (stability across bootstrap runs)
    - prev_matched : npos / (npos + nneg)  (prevalence of evaluation slice;
                     nneg = 0 for FunnyBirds so prev_matched = 1.0 always)
    - gap_norm     : gap_mean on [0,1] scale (conservative; gap already in [0,1])
    """
    out = pair_df.copy()
    out['gap_snr']  = out['gap_mean'] / out['gap_std'].replace(0, np.nan)
    denom           = (out['npos'] + out['nneg'].replace(0, np.nan))
    out['prev_matched'] = out['npos'] / denom
    out['gap_norm'] = out['gap_mean'] / 1.0
    return out


# Apply to both models' pairs DFs before any downstream use
if 'baseline' in results:
    _, pairs_base, _ = results['baseline']
    if not pairs_base.empty:
        pairs_base = add_gap_interpretability_cols(pairs_base)

if 'cbm' in results:
    _, pairs_cbm, _ = results['cbm']
    if not pairs_cbm.empty:
        pairs_cbm = add_gap_interpretability_cols(pairs_cbm)

print('add_gap_interpretability_cols applied.')

add_gap_interpretability_cols applied.


In [21]:
def summarize_by_attr(pairs_df: pd.DataFrame):
    """
    Collapses per-(model, attr, species-pair) into per-(model, attr) summary.
    Output: gap_mean, gap_median, gap_max, n_pairs, frac_p_small, frac_ci_above0, gap_snr_mean.
    Mirrors recall.ipynb verbatim.
    """
    if pairs_df.empty:
        return pairs_df
    g   = pairs_df.groupby(['model', 'attr'], as_index=False)
    out = g.agg(
        gap_mean       = ('gap_mean', 'mean'),
        gap_median     = ('gap_mean', 'median'),
        gap_max        = ('gap_mean', 'max'),
        n_pairs        = ('gap_mean', 'size'),
        frac_p_small   = ('gap_p',    lambda s: float(np.mean(np.asarray(s) <= 0.05))),
        frac_ci_above0 = ('gap_ci_lo', lambda s: float(np.mean(np.asarray(s) > 0))),
        gap_snr_mean   = ('gap_snr',  'mean'),
    ).sort_values(['model', 'gap_mean'], ascending=[True, False])
    return out


all_pairs_for_summary = []
if 'baseline' in results and not pairs_base.empty:
    all_pairs_for_summary.append(pairs_base)
if 'cbm' in results and not pairs_cbm.empty:
    all_pairs_for_summary.append(pairs_cbm)

summary = pd.concat(
    [summarize_by_attr(p) for p in all_pairs_for_summary],
    ignore_index=True
) if all_pairs_for_summary else pd.DataFrame()

summary

,model,attr,gap_mean,gap_median,gap_max,n_pairs,frac_p_small,frac_ci_above0,gap_snr_mean
0,baseline_fb,tail_8,0.067857,0.00,0.2,28,0.464286,0.464286,NaN
1,baseline_fb,tail_4,0.050000,0.05,0.1,6,0.500000,0.500000,NaN
2,baseline_fb,tail_5,0.050000,0.00,0.2,28,0.250000,0.250000,NaN
3,baseline_fb,tail_1,0.040000,0.00,0.1,10,0.400000,0.400000,NaN
4,baseline_fb,wing_3,0.040000,0.00,0.1,10,0.400000,0.400000,NaN
5,baseline_fb,beak_0,0.038000,0.00,0.1,200,0.380000,0.380000,NaN
6,baseline_fb,eye_1,0.032500,0.00,0.1,120,0.325000,0.325000,NaN
7,baseline_fb,tail_2,0.028571,0.00,0.1,21,0.285714,0.285714,NaN
8,baseline_fb,tail_3,0.028571,0.00,0.1,21,0.285714,0.285714,NaN
9,baseline_fb,beak_3,0.028205,0.00,0.1,78,0.282051,0.282051,NaN


In [22]:
def top_pairs(pairs_df: pd.DataFrame, model: str, attr: str, k=10):
    """
    Returns top-k pairs by gap_mean for a given (model, attr).
    Mirrors recall.ipynb verbatim.
    """
    sub = pairs_df[(pairs_df['model'] == model) & (pairs_df['attr'] == attr)].copy()
    if sub.empty:
        return sub
    cols = ['species_A', 'species_B',
            'gap_mean', 'gap_ci_lo', 'gap_ci_hi', 'gap_p',
            'gap_std', 'gap_snr', 'gap_norm',
            'npos', 'nneg', 'n_runs']
    cols = [c for c in cols if c in sub.columns]
    return sub.sort_values('gap_mean', ascending=False).head(k)[cols]


for a in ATTR_LIST:
    print(f'\nAttribute: {a}')
    if 'baseline' in results and not pairs_base.empty:
        print('Baseline top pairs:')
        display(top_pairs(pairs_base, 'baseline_fb', a, k=10))
    if 'cbm' in results and not pairs_cbm.empty:
        print('CBM top pairs:')
        display(top_pairs(pairs_cbm, 'cbm_fb', a, k=10))


Attribute: tail_5
Baseline top pairs:


,species_A,species_B,gap_mean,gap_ci_lo,gap_ci_hi,gap_p,gap_std,gap_snr,gap_norm,npos,nneg,n_runs
0,funnybird_23,funnybird_48,0.2,0.2,0.2,0.0,0.0,NaN,0.2,10,0,100
2,funnybird_38,funnybird_48,0.2,0.2,0.2,0.0,0.0,NaN,0.2,10,0,100
3,funnybird_26,funnybird_48,0.2,0.2,0.2,0.0,0.0,NaN,0.2,10,0,100
4,funnybird_05,funnybird_48,0.2,0.2,0.2,0.0,0.0,NaN,0.2,10,0,100
5,funnybird_40,funnybird_48,0.2,0.2,0.2,0.0,0.0,NaN,0.2,10,0,100
6,funnybird_09,funnybird_48,0.2,0.2,0.2,0.0,0.0,NaN,0.2,10,0,100
1,funnybird_11,funnybird_48,0.2,0.2,0.2,0.0,0.0,NaN,0.2,10,0,100
22,funnybird_23,funnybird_38,0.0,0.0,0.0,2.0,0.0,NaN,0.0,10,0,100
19,funnybird_26,funnybird_05,0.0,0.0,0.0,2.0,0.0,NaN,0.0,10,0,100
20,funnybird_11,funnybird_09,0.0,0.0,0.0,2.0,0.0,NaN,0.0,10,0,100


CBM top pairs:


,species_A,species_B,gap_mean,gap_ci_lo,gap_ci_hi,gap_p,gap_std,gap_snr,gap_norm,npos,nneg,n_runs
0,funnybird_05,funnybird_09,0.0,0.0,0.0,2.0,0.0,NaN,0.0,10,0,100
1,funnybird_05,funnybird_48,0.0,0.0,0.0,2.0,0.0,NaN,0.0,10,0,100
26,funnybird_09,funnybird_48,0.0,0.0,0.0,2.0,0.0,NaN,0.0,10,0,100
25,funnybird_11,funnybird_05,0.0,0.0,0.0,2.0,0.0,NaN,0.0,10,0,100
24,funnybird_11,funnybird_09,0.0,0.0,0.0,2.0,0.0,NaN,0.0,10,0,100
23,funnybird_11,funnybird_23,0.0,0.0,0.0,2.0,0.0,NaN,0.0,10,0,100
22,funnybird_11,funnybird_26,0.0,0.0,0.0,2.0,0.0,NaN,0.0,10,0,100
21,funnybird_11,funnybird_38,0.0,0.0,0.0,2.0,0.0,NaN,0.0,10,0,100
20,funnybird_11,funnybird_40,0.0,0.0,0.0,2.0,0.0,NaN,0.0,10,0,100
19,funnybird_11,funnybird_48,0.0,0.0,0.0,2.0,0.0,NaN,0.0,10,0,100



Attribute: tail_1
Baseline top pairs:


...

,species_A,species_B,gap_mean,gap_ci_lo,gap_ci_hi,gap_p,gap_std,gap_snr,gap_norm,npos,nneg,n_runs
1326,funnybird_17,funnybird_33,0.0,0.0,0.0,2.0,0.0,NaN,0.0,10,0,100
1337,funnybird_34,funnybird_17,0.0,0.0,0.0,2.0,0.0,NaN,0.0,10,0,100
1345,funnybird_34,funnybird_33,0.0,0.0,0.0,2.0,0.0,NaN,0.0,10,0,100
1344,funnybird_34,funnybird_35,0.0,0.0,0.0,2.0,0.0,NaN,0.0,10,0,100
1343,funnybird_34,funnybird_37,0.0,0.0,0.0,2.0,0.0,NaN,0.0,10,0,100
1342,funnybird_34,funnybird_39,0.0,0.0,0.0,2.0,0.0,NaN,0.0,10,0,100
1341,funnybird_34,funnybird_43,0.0,0.0,0.0,2.0,0.0,NaN,0.0,10,0,100
1340,funnybird_35,funnybird_17,0.0,0.0,0.0,2.0,0.0,NaN,0.0,10,0,100
1339,funnybird_35,funnybird_33,0.0,0.0,0.0,2.0,0.0,NaN,0.0,10,0,100
1338,funnybird_35,funnybird_39,0.0,0.0,0.0,2.0,0.0,NaN,0.0,10,0,100



Attribute: tail_6
Baseline top pairs:


,species_A,species_B,gap_mean,gap_ci_lo,gap_ci_hi,gap_p,gap_std,gap_snr,gap_norm,npos,nneg,n_runs
1347,funnybird_14,funnybird_32,0.0,0.0,0.0,2.0,0.0,NaN,0.0,10,0,100
1348,funnybird_41,funnybird_14,0.0,0.0,0.0,2.0,0.0,NaN,0.0,10,0,100
1349,funnybird_41,funnybird_32,0.0,0.0,0.0,2.0,0.0,NaN,0.0,10,0,100
1350,funnybird_41,funnybird_44,0.0,0.0,0.0,2.0,0.0,NaN,0.0,10,0,100
1351,funnybird_41,funnybird_45,0.0,0.0,0.0,2.0,0.0,NaN,0.0,10,0,100
1352,funnybird_44,funnybird_14,0.0,0.0,0.0,2.0,0.0,NaN,0.0,10,0,100
1353,funnybird_44,funnybird_32,0.0,0.0,0.0,2.0,0.0,NaN,0.0,10,0,100
1354,funnybird_45,funnybird_14,0.0,0.0,0.0,2.0,0.0,NaN,0.0,10,0,100
1355,funnybird_45,funnybird_32,0.0,0.0,0.0,2.0,0.0,NaN,0.0,10,0,100
1356,funnybird_45,funnybird_44,0.0,0.0,0.0,2.0,0.0,NaN,0.0,10,0,100


CBM top pairs:


,species_A,species_B,gap_mean,gap_ci_lo,gap_ci_hi,gap_p,gap_std,gap_snr,gap_norm,npos,nneg,n_runs
1347,funnybird_14,funnybird_32,0.0,0.0,0.0,2.0,0.0,NaN,0.0,10,0,100
1348,funnybird_41,funnybird_14,0.0,0.0,0.0,2.0,0.0,NaN,0.0,10,0,100
1349,funnybird_41,funnybird_32,0.0,0.0,0.0,2.0,0.0,NaN,0.0,10,0,100
1350,funnybird_41,funnybird_44,0.0,0.0,0.0,2.0,0.0,NaN,0.0,10,0,100
1351,funnybird_41,funnybird_45,0.0,0.0,0.0,2.0,0.0,NaN,0.0,10,0,100
1352,funnybird_44,funnybird_14,0.0,0.0,0.0,2.0,0.0,NaN,0.0,10,0,100
1353,funnybird_44,funnybird_32,0.0,0.0,0.0,2.0,0.0,NaN,0.0,10,0,100
1354,funnybird_45,funnybird_14,0.0,0.0,0.0,2.0,0.0,NaN,0.0,10,0,100
1355,funnybird_45,funnybird_32,0.0,0.0,0.0,2.0,0.0,NaN,0.0,10,0,100
1356,funnybird_45,funnybird_44,0.0,0.0,0.0,2.0,0.0,NaN,0.0,10,0,100


In [23]:
if 'baseline' in results:
    _, _, species_base = results['baseline']
    print('Baseline species table (top 30 by TP):')
    display(species_base.sort_values('tp', ascending=False).head(30))

if 'cbm' in results:
    _, _, species_cbm = results['cbm']
    print('CBM species table (top 30 by TP):')
    display(species_cbm.sort_values('tp', ascending=False).head(30))

Baseline species table (top 30 by TP):


,species_id,species_name,n,n_pos,n_pred_pos,n_neg,prevalence,tp,recall,precision,recall_bs_mean,recall_ci_lo,recall_ci_hi,recall_ci_width,B,attr,model
213,1,funnybird_01,10,10,10,0,1.0,10,1.0,1.0,1.0,1.0,1.0,0.0,100,beak_3,baseline_fb
591,15,funnybird_15,10,10,10,0,1.0,10,1.0,1.0,1.0,1.0,1.0,0.0,100,beak_1,baseline_fb
1032,6,funnybird_06,10,10,10,0,1.0,10,1.0,1.0,1.0,1.0,1.0,0.0,100,foot_1,baseline_fb
789,13,funnybird_13,10,10,10,0,1.0,10,1.0,1.0,1.0,1.0,1.0,0.0,100,wing_1,baseline_fb
279,3,funnybird_03,10,10,10,0,1.0,10,1.0,1.0,1.0,1.0,1.0,0.0,100,beak_0,baseline_fb
791,15,funnybird_15,10,10,10,0,1.0,10,1.0,1.0,1.0,1.0,1.0,0.0,100,wing_1,baseline_fb
519,44,funnybird_44,10,10,10,0,1.0,10,1.0,1.0,1.0,1.0,1.0,0.0,100,eye_2,baseline_fb
276,24,funnybird_24,10,10,10,0,1.0,10,1.0,1.0,1.0,1.0,1.0,0.0,100,beak_0,baseline_fb
1117,42,funnybird_42,10,10,10,0,1.0,10,1.0,1.0,1.0,1.0,1.0,0.0,100,tail_0,baseline_fb
155,30,funnybird_30,10,10,10,0,1.0,10,1.0,1.0,1.0,1.0,1.0,0.0,100,eye_0,baseline_fb


CBM species table (top 30 by TP):


,species_id,species_name,n,n_pos,n_pred_pos,n_neg,prevalence,tp,recall,precision,recall_bs_mean,recall_ci_lo,recall_ci_hi,recall_ci_width,B,attr,model
213,1,funnybird_01,10,10,10,0,1.0,10,1.0,1.0,1.0,1.0,1.0,0.0,100,beak_3,cbm_fb
615,40,funnybird_40,10,10,10,0,1.0,10,1.0,1.0,1.0,1.0,1.0,0.0,100,wing_4,cbm_fb
144,18,funnybird_18,10,10,10,0,1.0,10,1.0,1.0,1.0,1.0,1.0,0.0,100,tail_8,cbm_fb
786,10,funnybird_10,10,10,10,0,1.0,10,1.0,1.0,1.0,1.0,1.0,0.0,100,wing_1,cbm_fb
591,15,funnybird_15,10,10,10,0,1.0,10,1.0,1.0,1.0,1.0,1.0,0.0,100,beak_1,cbm_fb
517,42,funnybird_42,10,10,10,0,1.0,10,1.0,1.0,1.0,1.0,1.0,0.0,100,eye_2,cbm_fb
518,43,funnybird_43,10,10,10,0,1.0,10,1.0,1.0,1.0,1.0,1.0,0.0,100,eye_2,cbm_fb
789,13,funnybird_13,10,10,10,0,1.0,10,1.0,1.0,1.0,1.0,1.0,0.0,100,wing_1,cbm_fb
1032,6,funnybird_06,10,10,10,0,1.0,10,1.0,1.0,1.0,1.0,1.0,0.0,100,foot_1,cbm_fb
791,15,funnybird_15,10,10,10,0,1.0,10,1.0,1.0,1.0,1.0,1.0,0.0,100,wing_1,cbm_fb


In [24]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

if 'baseline' in results and 'cbm' in results:
    info_b, _, _ = results['baseline']
    info_c, _, _ = results['cbm']

    merged = (
        info_b[['attr', 'mean_gap', 'test_acc']]
        .rename(columns={'mean_gap': 'gap_base', 'test_acc': 'acc_base'})
        .merge(
            info_c[['attr', 'mean_gap', 'test_acc']]
            .rename(columns={'mean_gap': 'gap_cbm', 'test_acc': 'acc_cbm'}),
            on='attr', how='outer')
    )

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    # Left: recall gap scatter
    ax = axes[0]
    ax.scatter(merged['gap_base'], merged['gap_cbm'], alpha=0.7, s=60)
    for _, row in merged.dropna().iterrows():
        ax.annotate(row['attr'], (row['gap_base'], row['gap_cbm']), fontsize=6, alpha=0.6)
    lim = max(merged[['gap_base', 'gap_cbm']].max().max(), 0.05) * 1.1
    ax.plot([0, lim], [0, lim], 'k--', alpha=0.4, label='y=x')
    ax.set_xlabel('Baseline recall gap')
    ax.set_ylabel('CBM recall gap')
    ax.set_title('Recall gap: Baseline vs CBM\n(above diagonal = CBM more entangled)')
    ax.legend(); ax.grid(True, alpha=0.3)

    # Right: probe accuracy scatter
    ax = axes[1]
    ax.scatter(merged['acc_base'], merged['acc_cbm'], alpha=0.7, s=60)
    for _, row in merged.dropna().iterrows():
        ax.annotate(row['attr'], (row['acc_base'], row['acc_cbm']), fontsize=6, alpha=0.6)
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.4, label='y=x')
    ax.set_xlabel('Baseline probe accuracy')
    ax.set_ylabel('CBM probe accuracy')
    ax.set_title('Probe accuracy: Baseline vs CBM\n(above diagonal = CBM more concept-linear)')
    ax.legend(); ax.grid(True, alpha=0.3)

    plt.suptitle('FunnyBirds: Baseline vs CBM', y=1.02)
    plt.tight_layout()
    plt.savefig('fb_baseline_vs_cbm.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved fb_baseline_vs_cbm.png')

Saved fb_baseline_vs_cbm.png


In [25]:
if 'baseline' in results and 'cbm' in results:
    info_b, _, _ = results['baseline']
    info_c, _, _ = results['cbm']

    merged2 = (
        info_b[['attr', 'mean_gap']].rename(columns={'mean_gap': 'gap_base'})
        .merge(info_c[['attr', 'mean_gap']].rename(columns={'mean_gap': 'gap_cbm'}),
               on='attr', how='outer')
        .sort_values('gap_base', ascending=False)
    )

    x     = np.arange(len(merged2))
    width = 0.35
    fig, ax = plt.subplots(figsize=(max(10, len(merged2) * 0.6), 5))
    ax.bar(x - width/2, merged2['gap_base'], width, label='Baseline', alpha=0.8)
    ax.bar(x + width/2, merged2['gap_cbm'],  width, label='CBM',      alpha=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(merged2['attr'], rotation=45, ha='right', fontsize=8)
    ax.set_ylabel('Mean recall gap')
    ax.set_title('FunnyBirds: Mean recall gap per concept\n(lower = less species-identity entanglement)')
    ax.legend(); ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.savefig('fb_gap_per_concept.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved fb_gap_per_concept.png')

elif len(results) == 1:
    name = list(results.keys())[0]
    info, _, _ = results[name]
    fig, ax = plt.subplots(figsize=(max(8, len(info) * 0.5), 4))
    info_s = info.sort_values('mean_gap', ascending=False)
    ax.bar(np.arange(len(info_s)), info_s['mean_gap'], alpha=0.8)
    ax.set_xticks(np.arange(len(info_s)))
    ax.set_xticklabels(info_s['attr'], rotation=45, ha='right', fontsize=8)
    ax.set_ylabel('Mean recall gap')
    ax.set_title(f'FunnyBirds {name}: Mean recall gap per concept')
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.savefig(f'fb_{name}_gap_per_concept.png', dpi=150, bbox_inches='tight')
    plt.show()

Saved fb_gap_per_concept.png


In [26]:
if 'baseline' in results and 'cbm' in results:
    info_b, _, _ = results['baseline']
    info_c, _, _ = results['cbm']

    def compute_frac_p90_jump(info_df, pairs_df, model_name):
        """
        For each concept:
          frac_p90 = fraction of pairs whose gap_mean >= the 90th-pct threshold
                     of that concept's own gap_mean distribution
          jump     = p90_gap from info_df (worst-10% pair gap for this concept)
        Mirrors the frac_p90 / jump analysis from recall.ipynb.
        """
        rows = []
        for _, row in info_df.iterrows():
            attr   = row['attr']
            p90    = float(row['p90_gap']) if not np.isnan(row['p90_gap']) else np.nan
            spread = attr_to_spread.get(attr, np.nan)

            if pairs_df.empty or 'attr' not in pairs_df.columns \
                    or attr not in pairs_df['attr'].values:
                frac_p90 = np.nan
            else:
                sub      = pairs_df[pairs_df['attr'] == attr]
                thr_val  = np.nanquantile(sub['gap_mean'].values, 0.9) if len(sub) > 0 else np.nan
                frac_p90 = float(np.mean(sub['gap_mean'].values >= thr_val)) \
                           if (len(sub) > 0 and not np.isnan(thr_val)) else np.nan

            rows.append({'attr': attr, 'model': model_name,
                         'frac_p90': frac_p90, 'jump': p90, 'spread': spread})
        return pd.DataFrame(rows)

    _pb = pairs_base if not pairs_base.empty else pd.DataFrame()
    _pc = pairs_cbm  if not pairs_cbm.empty  else pd.DataFrame()
    fp_base = compute_frac_p90_jump(info_b, _pb, 'baseline_fb')
    fp_cbm  = compute_frac_p90_jump(info_c, _pc, 'cbm_fb')

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for ax, fp, title in zip(axes,
                              [fp_base, fp_cbm],
                              ['Baseline', 'CBM']):
        sc = ax.scatter(fp['frac_p90'], fp['jump'], alpha=0.8, s=60,
                        c=fp['spread'], cmap='viridis')
        for _, r in fp.dropna().iterrows():
            ax.annotate(r['attr'], (r['frac_p90'], r['jump']), fontsize=6, alpha=0.7)
        ax.set_xlabel('frac_p90 (fraction of pairs at or above 90th-pct gap)')
        ax.set_ylabel('jump (p90_gap — worst-10% pair gap for this concept)')
        ax.set_title(f'FunnyBirds {title}: frac_p90 vs jump')
        plt.colorbar(sc, ax=ax, label='recall spread (p90−p10)')
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('fb_frac_p90_vs_jump.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved fb_frac_p90_vs_jump.png')

Saved fb_frac_p90_vs_jump.png


ModuleNotFoundError: No module named 'openai'